# A/B Testing Toolkit: Walkthrough

This notebook works through a small CTR experiment. CTR = clicks / impressions.

We compare a control banner to a redesigned banner.

In [ ]:
import sys
sys.path.append('..')

from src import (two_proportion_ztest, welch_ttest,
                 prob_b_beats_a, credible_interval, expected_loss,
                 proportion_sample_size, continuous_sample_size)
from src.sim import generate_binary

## Pre-experiment: how many users do we need?

Baseline CTR is 5%, we want to detect a 0.5pp lift at 80% power, alpha=0.05.

In [ ]:
n = proportion_sample_size(baseline_rate=0.05, mde=0.005, power=0.8, alpha=0.05)
print(f'per-arm n = {n}')

## Run the experiment (simulated)

We simulate `n` users per arm with a true rate of 5% in control and 5.6% in treatment.

In [ ]:
df = generate_binary(n_per_arm=n, p_a=0.05, p_b=0.056, seed=7)
df.groupby('variant')['converted'].agg(['sum', 'count', 'mean'])

## Frequentist analysis

In [ ]:
g = df.groupby('variant')['converted']
succ_a, n_a = int(g.sum()['A']), int(g.count()['A'])
succ_b, n_b = int(g.sum()['B']), int(g.count()['B'])
res = two_proportion_ztest(succ_a, n_a, succ_b, n_b)
res